# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md`, `CHEATSHEETS/gpu-zugang.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | 2025-05-15 |
| Modell | `Qwen/Qwen2.5-7B-Instruct` |
| Server | gauss |
| GPU-Index | 0 (V100 SXM2, 32 GB) |
| Schema-Datei | `SCHEMA.md` |
| Aktueller Run-Tag | `baseline` |
| Predictions-Datei | `predictions.jsonl` |
| Truncation | 2000 Zeichen (Head-Truncation) |

Bei jeder neuen Iteration: Run-Tag + Predictions-Datei + Datum aktualisieren.

## Phase 3 — Pipeline bauen + Baseline-Inferenz

In [1]:
# GPU pruefen — IMMER zuerst, VOR dem ersten Modell-Load
import os
# gauss: freie GPU-Nummer aus `nvidia-smi` waehlen (0-3). euler: 0 (nur eine GPU).
GPU_INDEX = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_INDEX

import torch
print(f'CUDA verfuegbar: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA verfuegbar: True
GPU: Tesla V100-SXM2-32GB
VRAM: 33.8 GB


In [2]:
# 7B-Modell laden — float32 (V100-Constraint), NICHT fp16/bf16, NICHT device_map='auto'
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

if 'model' in globals():
    print('7B-Modell ist bereits geladen — ueberspringe (Doppel-Load = OOM vermeiden).')
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
    ).to('cuda').eval()
    print('7B-Modell geladen.')
print(f'Belegt: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')

/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

7B-Modell geladen.
Belegt: 31.6 GB


In [2]:
# System-Prompt mit Schema (Baseline)
SYSTEM_PROMPT = '''Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.
Antworte NUR mit einem JSON-Objekt — kein Text davor oder danach.

Schema (alle Felder pflicht):
{"refnr": "<string>", "homeoffice": ..., "vertragsart": ..., "erfahrungslevel": ...,
 "gehalt_min_eur": <int|null>, "gehalt_zeitraum": <"monat"|"jahr"|null>,
 "skills_top3": ["skill1", "skill2", "skill3"]}

Erlaubte Werte:
- homeoffice: ja | teilweise | nein | remote | nicht_genannt
- vertragsart: ausbildung | festanstellung | praktikum | werkstudent | sonstiges
- erfahrungslevel: junior | mid | senior | egal | nicht_genannt

Definitionen:
- remote: 100% ortsunabhaengig
- teilweise: explizites Hybrid mit Wochentagen oder Verhaeltnis
- ja: HO angeboten, Modus nicht spezifiziert
- junior: 2 Jahre Berufserfahrung oder Berufseinsteiger
- mid: 2-5 Jahre; senior: 5+ Jahre; egal: alle Stufen willkommen
- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag
- skills_top3: max 3 Tools/Technologien, lowercase

Beispiel:
User: refnr: 10000-1002-S\nData Analyst Hamburg, 2 Jahre, SQL/Python/Power BI, 2 Tage HO, 52k-62k EUR, Festanstellung.
Assistant: {"refnr": "10000-1002-S", "homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 52000, "gehalt_zeitraum": "jahr", "skills_top3": ["sql", "python", "power_bi"]}'''.strip()

print('System-Prompt definiert.')

System-Prompt definiert.


In [3]:
import re, json, time
from pathlib import Path

def extract_json(response):
    """Robust: entfernt Markdown-Codefences und sucht das erste balancierte {...}-Objekt."""
    text = response.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None


def run_pipeline(anzeigen, output_path, prompt=None, max_chars=2000):
    used_prompt = prompt or SYSTEM_PROMPT
    parse_fails = 0
    t0 = time.time()
    with open(output_path, 'w', encoding='utf-8') as out:
        for i, anzeige in enumerate(anzeigen, 1):
            text_trunc = anzeige['text'][:max_chars]
            messages = [
                {'role': 'system', 'content': used_prompt},
                {'role': 'user',   'content': f"refnr: {anzeige['refnr']}\n{text_trunc}"},
            ]
            prompt_str = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt_str, return_tensors='pt').to('cuda')
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=200, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True,
            )
            result = extract_json(response)
            if result is None:
                parse_fails += 1
                print(f'  [{i:02d}] PARSE-FAIL: {anzeige["refnr"]}')
                result = {'refnr': anzeige['refnr'], '_parse_fail': True}
            else:
                print(f'  [{i:02d}] OK: {anzeige["refnr"]}')
            out.write(json.dumps(result, ensure_ascii=False) + '\n')
    elapsed = time.time() - t0
    print(f'\nFertig: {len(anzeigen)} Anzeigen, {elapsed:.0f}s ({elapsed/len(anzeigen):.1f}s/Anzeige)')
    print(f'Parse-Fails: {parse_fails}/{len(anzeigen)}')
    return parse_fails

print('Pipeline-Funktion definiert.')

Pipeline-Funktion definiert.


In [4]:
# Baseline-Run — nur die 12 Hand-Gold-Anzeigen
GOLD_IDS = [
    '10000-1001-S', '10000-1002-S', '10000-1003-S', '10000-1004-S',
    '10000-1005-S', '10000-1006-S', '10000-1007-S', '10000-1008-S',
    '10000-1009-S', '10000-1010-S', '10000-1011-S', '10000-1012-S',
]

with open('../daten/eigener_korpus.jsonl', encoding='utf-8') as f:
    alle = [json.loads(line) for line in f]

gold_anzeigen = [a for a in alle if a['refnr'] in GOLD_IDS]
print(f'{len(gold_anzeigen)} Anzeigen fuer Baseline.')
run_pipeline(gold_anzeigen, output_path='../predictions.jsonl')

12 Anzeigen fuer Baseline.


NameError: name 'tokenizer' is not defined

## Phase 4 — Iteration A

**Run-Header Iteration A**

| Feld | Wert |
|---|---|
| Datum | 2025-05-15 |
| Aktueller Run-Tag | `iter_A` |
| Predictions-Datei | `predictions_iter_A.jsonl` |
| Änderung | Prompt-Klarstellung `homeoffice`: `ja` vs. `teilweise` operationalisiert |

In [5]:
# Iteration A — Prompt-Klarstellung homeoffice
# Diagnose: Grenze ja/teilweise zu unscharf -> Modell waehlt teilweise bei unspez. HO
SYSTEM_PROMPT_A = SYSTEM_PROMPT.replace(
    '- ja: HO angeboten, Modus nicht spezifiziert',
    '- ja: HO angeboten, ABER kein Verhaeltnis oder Wochentage genannt\n'
    '- teilweise: Hybrid MIT explizitem Verhaeltnis ODER Wochentagen (z.B. 2 Tage Buero, 60% remote)'
)

run_pipeline(gold_anzeigen, output_path='../predictions_iter_A.jsonl', prompt=SYSTEM_PROMPT_A)

NameError: name 'tokenizer' is not defined

## Phase 4 — Iteration B

**Run-Header Iteration B**

| Feld | Wert |
|---|---|
| Datum | 2025-05-15 |
| Aktueller Run-Tag | `iter_B` |
| Predictions-Datei | `predictions_iter_B.jsonl` |
| Änderung | Prompt-Ergänzung `gehalt_min_eur`: Stundenlöhne/Tagessätze → null |

In [6]:
# Iteration B — aufbauend auf Iter. A, Gehalt-Klarstellung ergaenzt
SYSTEM_PROMPT_B = SYSTEM_PROMPT_A.replace(
    '- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag',
    '- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag\n'
    '  WICHTIG: Stundenloehne (EUR/h) und Tagessaetze sind KEIN Monats-/Jahresgehalt -> null setzen'
)

run_pipeline(gold_anzeigen, output_path='../predictions_iter_B.jsonl', prompt=SYSTEM_PROMPT_B)

NameError: name 'tokenizer' is not defined

## Phase 6 — Voller 7B-Run (gauss)

**Run-Header Phase 6**

| Feld | Wert |
|---|---|
| Datum | 2025-05-16 |
| Aktueller Run-Tag | `full_7b` |
| Predictions-Datei | `predictions_7b_full.jsonl` |
| Datenquelle | voller Korpus (32 Anzeigen) |
| Prompt | finale Version aus Iteration B |

In [9]:
# Voller 7B-Run mit finaler Pipeline (Prompt aus Iter. B)
print(f'Voller Korpus: {len(alle)} Anzeigen')
run_pipeline(alle, output_path='../predictions_7b_full.jsonl', prompt=SYSTEM_PROMPT_B)

# Schema-Konformitaet auf dem vollen Output pruefen
import subprocess
r = subprocess.run(
    ['python', 'annotation/validate.py', '--validate-jsonl', 'predictions_7b_full.jsonl'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

Voller Korpus: 32 Anzeigen
  [01] OK: 10000-1001-S
  [02] OK: 10000-1002-S
  [03] OK: 10000-1003-S
  [04] OK: 10000-1004-S
  [05] OK: 10000-1005-S
  [06] OK: 10000-1006-S
  [07] OK: 10000-1007-S
  [08] OK: 10000-1008-S
  [09] OK: 10000-1009-S
  [10] OK: 10000-1010-S
  [11] OK: 10000-1011-S
  [12] OK: 10000-1012-S
  [13] OK: 10000-1013-S
  [14] OK: 10000-1014-S
  [15] OK: 10000-1015-S
  [16] OK: 10000-1016-S
  [17] OK: 10000-1017-S
  [18] OK: 10000-1018-S
  [19] OK: 10000-1019-S
  [20] OK: 10000-1020-S
  [21] OK: 10000-1021-S
  [22] OK: 10000-1022-S
  [23] OK: 10000-1023-S
  [24] OK: 10000-1024-S
  [25] OK: 10000-1025-S
  [26] OK: 10000-1026-S
  [27] OK: 10000-1027-S
  [28] OK: 10000-1028-S
  [29] OK: 10000-1029-S
  [30] OK: 10000-1030-S
  [31] OK: 10000-1031-S
  [32] OK: 10000-1032-S

Fertig: 32 Anzeigen, 157s (4.9s/Anzeige)
Parse-Fails: 0/32

JSONL Schema-Check: predictions_7b_full.jsonl
Geprüfte Zeilen: 32
JSON-Parse-Fails: 0
Keine Feld-Verletzungen.



## Phase 6 — 3B-Run (euler)

> **Auf euler:** Kernel neu starten, dann oben **nur** die GPU-Check-Zelle ausfuehren und hier weiter — die 7B-Lade-/Run-Zellen NICHT (7B passt nicht in 16 GB). Die folgende Vorbereitungs-Zelle ist self-contained (Korpus + Prompt + Parser), du brauchst die 7B-Zellen also nicht.

**Run-Header 3B**

| Feld | Wert |
|---|---|
| Datum | 2025-05-16 |
| Modell | `Qwen/Qwen2.5-3B-Instruct` |
| Server | euler |
| GPU-Index | 0 (V100 SXM2, 16 GB) |
| Aktueller Run-Tag | `full_3b` |
| Predictions-Datei | `predictions_3b_full.jsonl` |

In [2]:
# === euler-3B-Vorbereitung (self-contained) ===
import re, json, time
from pathlib import Path

with open('../daten/eigener_korpus.jsonl', encoding='utf-8') as f:
    alle = [json.loads(line) for line in f if line.strip()]
print(f'Korpus: {len(alle)} Anzeigen')

# extract_json sicherstellen (falls die 7B-Zellen nicht gelaufen sind)
if 'extract_json' not in globals():
    def extract_json(response):
        text = response.strip()
        text = re.sub(r'^```(?:json)?', '', text).strip()
        text = re.sub(r'```$', '', text).strip()
        start = text.find('{')
        if start == -1:
            return None
        depth = 0
        for i in range(start, len(text)):
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        return None
        return None

# Finaler Prompt aus Iteration B — identisch zur 7B-Pipeline (hier inline, damit
# die euler-Sektion ohne die 7B-Zellen lauffaehig ist).
SYSTEM_PROMPT_B = "Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.\nAntworte NUR mit einem JSON-Objekt — kein Text davor oder danach.\n\nSchema (alle Felder pflicht):\n{\"refnr\": \"<string>\", \"homeoffice\": ..., \"vertragsart\": ..., \"erfahrungslevel\": ...,\n \"gehalt_min_eur\": <int|null>, \"gehalt_zeitraum\": <\"monat\"|\"jahr\"|null>,\n \"skills_top3\": [\"skill1\", \"skill2\", \"skill3\"]}\n\nErlaubte Werte:\n- homeoffice: ja | teilweise | nein | remote | nicht_genannt\n- vertragsart: ausbildung | festanstellung | praktikum | werkstudent | sonstiges\n- erfahrungslevel: junior | mid | senior | egal | nicht_genannt\n\nDefinitionen:\n- remote: 100% ortsunabhaengig\n- teilweise: explizites Hybrid mit Wochentagen oder Verhaeltnis\n- ja: HO angeboten, ABER kein Verhaeltnis oder Wochentage genannt\n- teilweise: Hybrid MIT explizitem Verhaeltnis ODER Wochentagen (z.B. 2 Tage Buero, 60% remote)\n- junior: 2 Jahre Berufserfahrung oder Berufseinsteiger\n- mid: 2-5 Jahre; senior: 5+ Jahre; egal: alle Stufen willkommen\n- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag\n  WICHTIG: Stundenloehne (EUR/h) und Tagessaetze sind KEIN Monats-/Jahresgehalt -> null setzen\n- skills_top3: max 3 Tools/Technologien, lowercase\n\nBeispiel:\nUser: refnr: 10000-1002-S\nData Analyst Hamburg, 2 Jahre, SQL/Python/Power BI, 2 Tage HO, 52k-62k EUR, Festanstellung.\nAssistant: {\"refnr\": \"10000-1002-S\", \"homeoffice\": \"teilweise\", \"vertragsart\": \"festanstellung\", \"erfahrungslevel\": \"mid\", \"gehalt_min_eur\": 52000, \"gehalt_zeitraum\": \"jahr\", \"skills_top3\": [\"sql\", \"python\", \"power_bi\"]}"
print('Vorbereitung ok: Korpus, extract_json und SYSTEM_PROMPT_B vorhanden.')

Korpus: 32 Anzeigen
Vorbereitung ok: Korpus, extract_json und SYSTEM_PROMPT_B vorhanden.


In [3]:
# Frischer Kernel-Check (euler): hier sollte KEIN 7B-Modell im Speicher sein.
print('7B im Speicher:', 'model' in globals())
print('Belegt:', f'{torch.cuda.memory_allocated(0)/1e9:.1f} GB' if torch.cuda.is_available() else 'keine GPU')

7B im Speicher: False
Belegt: 0.0 GB


In [4]:
# 3B-Modell laden — nur auf euler. float32 wie im Cheatsheet vorgegeben.
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

MODEL_NAME_3B = 'Qwen/Qwen2.5-3B-Instruct'

def _load_3b():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME_3B)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_3B, torch_dtype=torch.float32,
    ).to('cuda').eval()
    return tok, mdl

if 'model_3b' in globals():
    print('3B bereits geladen — ueberspringe.')
else:
    try:
        tokenizer_3b, model_3b = _load_3b()
        print('3B-Modell geladen (float32).')
    except ValueError as e:
        if 'GPTQ' in str(e) or 'quantiz' in str(e).lower():
            # defekter/quantisierter Cache-Eintrag -> EINMALIG sauberen float32-Build ziehen
            print('GPTQ-Cache erkannt — ziehe einmalig einen sauberen float32-Build (~6 GB)...')
            snapshot_download(MODEL_NAME_3B, force_download=True)
            tokenizer_3b, model_3b = _load_3b()
            print('3B-Modell geladen (float32, Cache repariert).')
        else:
            raise
print(f'Belegt: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')

/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

3B-Modell geladen (float32).
Belegt: 13.6 GB


In [5]:
# 3B-Inferenz-Schleife
parse_fails_3b = 0
t0 = time.time()

with open('../predictions_3b_full.jsonl', 'w', encoding='utf-8') as out:
    for i, anzeige in enumerate(alle, 1):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT_B},
            {'role': 'user',   'content': f"refnr: {anzeige['refnr']}\n{anzeige['text'][:2000]}"},
        ]
        prompt_str = tokenizer_3b.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer_3b(prompt_str, return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = model_3b.generate(
                **inputs, max_new_tokens=200, do_sample=False,
                pad_token_id=tokenizer_3b.eos_token_id,
            )
        response = tokenizer_3b.decode(
            outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        result = extract_json(response)
        if result is None:
            parse_fails_3b += 1
            result = {'refnr': anzeige['refnr'], '_parse_fail': True}
        out.write(json.dumps(result, ensure_ascii=False) + '\n')
        if i % 8 == 0:
            print(f'  {i}/{len(alle)} ...')

print(f'3B-Run: {len(alle)} Anzeigen in {time.time()-t0:.0f}s, Parse-Fails: {parse_fails_3b}')

# Schema-Konformitaet pruefen
import subprocess
r = subprocess.run(
    ['python', 'annotation/validate.py', '--validate-jsonl', 'predictions_3b_full.jsonl'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  8/32 ...
  16/32 ...
  24/32 ...
  32/32 ...
3B-Run: 32 Anzeigen in 156s, Parse-Fails: 1

JSONL Schema-Check: predictions_3b_full.jsonl
Geprüfte Zeilen: 32
JSON-Parse-Fails: 0

11 Feld-Verletzungen (sortiert):
     2×  vertragsart: 'aufbaustudiengang' nicht in Schema
     2×  skills_top3: 4 Einträge (max 3)
     1×  vertragsart: 'aufbauend_ausbildung' nicht in Schema
     1×  vertragsart: 'trainee' nicht in Schema
     1×  vertragsart: 'aufbauendes_ausbildung' nicht in Schema
     1×  homeoffice: leer
     1×  vertragsart: leer
     1×  erfahrungslevel: leer
     1×  vertragsart: 'aufbauende_ausbildung' nicht in Schema



In [7]:
# Nur die Prompt-Definitionen, ohne Modell/Pipeline
SYSTEM_PROMPT_A = SYSTEM_PROMPT.replace(
    '- ja: HO angeboten, Modus nicht spezifiziert',
    '- ja: HO angeboten, ABER kein Verhaeltnis oder Wochentage genannt\n'
    '- teilweise: Hybrid MIT explizitem Verhaeltnis ODER Wochentagen (z.B. 2 Tage Buero, 60% remote)'
)
SYSTEM_PROMPT_B = SYSTEM_PROMPT_A.replace(
    '- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag',
    '- gehalt_min_eur: untere Zahl einer Range als ganze Zahl; null wenn kein Betrag\n'
    '  WICHTIG: Stundenloehne (EUR/h) und Tagessaetze sind KEIN Monats-/Jahresgehalt -> null setzen'
)
print(SYSTEM_PROMPT_B)

Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.
Antworte NUR mit einem JSON-Objekt — kein Text davor oder danach.

Schema (alle Felder pflicht):
{"refnr": "<string>", "homeoffice": ..., "vertragsart": ..., "erfahrungslevel": ...,
 "gehalt_min_eur": <int|null>, "gehalt_zeitraum": <"monat"|"jahr"|null>,
 "skills_top3": ["skill1", "skill2", "skill3"]}

Erlaubte Werte:
- homeoffice: ja | teilweise | nein | remote | nicht_genannt
- vertragsart: ausbildung | festanstellung | praktikum | werkstudent | sonstiges
- erfahrungslevel: junior | mid | senior | egal | nicht_genannt

Definitionen:
- remote: 100% ortsunabhaengig
- teilweise: explizites Hybrid mit Wochentagen oder Verhaeltnis
- ja: HO angeboten, ABER kein Verhaeltnis oder Wochentage genannt
- teilweise: Hybrid MIT explizitem Verhaeltnis ODER Wochentagen (z.B. 2 Tage Buero, 60% remote)
- junior: 2 Jahre Berufserfahrung oder Berufseinsteiger
- mid: 2-5 Jahre; senior: 5+ Jahre; egal: alle Stufen willkommen
- gehal

In [1]:
import json, csv
corpus = {json.loads(l)['refnr']: json.loads(l)
          for l in open('../daten/eigener_korpus.jsonl') if l.strip()}
gold = {r['id']: r for r in csv.DictReader(open('../annotation/meine_gold.csv'))}

def few_shot(rid):
    g = gold[rid]
    obj = {"id": rid, "homeoffice": g['homeoffice'], "vertragsart": g['vertragsart'],
           "erfahrungslevel": g['erfahrungslevel'],
           "gehalt_min_eur": int(g['gehalt_min_eur']) if g['gehalt_min_eur'] else None,
           "gehalt_zeitraum": g['gehalt_zeitraum'] or None,
           "skills_top3": [s for s in g['skills_top3'].split('|') if s]}
    print(f"=== id: {rid} ===\n{corpus[rid]['text']}\n-> {json.dumps(obj, ensure_ascii=False)}\n")

for rid in ['10000-1001-S', '10000-1002-S', '10000-1011-S']:
    few_shot(rid)

=== id: 10000-1001-S ===
Die Stadtwerke Bremen AG bietet zum 1. September eine Ausbildung zum/zur Fachinformatiker/in Fachrichtung Daten- und Prozessanalyse an. Während der dreijährigen Ausbildung lernst du, Daten zu erheben, aufzubereiten und auszuwerten. Du arbeitest mit Python, SQL und Excel. Außerdem lernst du, Geschäftsprozesse zu modellieren und zu optimieren. Voraussetzung: mittlerer Schulabschluss oder Abitur. Ausbildungsvergütung nach Tarif (1. Jahr: 1.050 EUR, 2. Jahr: 1.100 EUR, 3. Jahr: 1.200 EUR). Arbeitsort: Bremen, Präsenz erforderlich.
-> {"id": "10000-1001-S", "homeoffice": "nein", "vertragsart": "ausbildung", "erfahrungslevel": "junior", "gehalt_min_eur": 1050, "gehalt_zeitraum": "monat", "skills_top3": ["python", "sql", "excel"]}

=== id: 10000-1002-S ===
Wir suchen zur Verstärkung unseres Teams einen Data Analyst (m/w/d) in Hamburg. Sie analysieren Bestands- und Transportdaten, erstellen regelmäßige Reports in Power BI und unterstützen das Management mit datengetrie